# 第 3 週 實作｜Cauchy 均值定理、L'Hôpital 的證明與泰勒多項式

銜接課用了羅必達一整週卻沒說它為什麼成立。這週補上證明,順便拆穿一個大家都在犯的循環論證,最後用泰勒多項式把「近似」變成「有保證的近似」。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜泰勒多項式:一階一階逼近上去

把 $P_1,P_3,P_5,P_7,P_9$ 疊在 $\sin x$ 上,親眼看它們怎麼一段一段「咬」住原函數。


In [ ]:
x = sp.Symbol('x')

def taylor_poly(f, n, a=0):
    """回傳 f 在 a 的 n 階泰勒多項式(sympy 運算式)"""
    return sum(sp.diff(f, x, k).subs(x, a) / sp.factorial(k) * (x - a)**k
               for k in range(n + 1))

xs = np.linspace(-8, 8, 600)
plt.plot(xs, np.sin(xs), 'k', lw=2.5, label='sin x')
for n in [1, 3, 5, 7, 9]:
    P = taylor_poly(sp.sin(x), n)
    fn = sp.lambdify(x, P, 'numpy')
    plt.plot(xs, fn(xs), lw=1.2, label=f'P{n}')
    print(f"P{n} =", sp.expand(P))
plt.ylim(-2, 2); plt.legend(ncol=3, fontsize=8)
plt.title('Taylor polynomials of sin x at a = 0')
plt.show()

In [ ]:
# TODO 學生練習:把 sp.sin(x) 換成 sp.exp(x),範圍改 np.linspace(-3, 3, 400)
# exp 的泰勒多項式和 sin 的行為有什麼不同?(提示:看有沒有正負震盪)

## Lab 2｜餘項界是真的嗎

觀念 7 說誤差不超過 $\dfrac{|x|^{n+1}}{(n+1)!}$。這格把「界」和「實際誤差」畫在一起,檢查界有沒有被突破、又有多緊。


In [ ]:
from math import factorial

def taylor_sin(x0, n):
    """sin 在 0 的 n 階泰勒多項式,直接用級數係數"""
    return sum((-1)**k * x0**(2*k+1) / factorial(2*k+1)
               for k in range((n + 1) // 2))

print(f"{'x':>6} {'n':>3} {'P_n(x)':>14} {'sin(x)':>14} {'實際誤差':>12} {'餘項界':>12} {'界成立?':>8}")
for x0 in [0.5, 1.0, 2.0]:
    for n in [3, 5, 7]:
        approx = taylor_sin(x0, n)
        exact  = math.sin(x0)
        err    = abs(approx - exact)
        bound  = abs(x0)**(n+1) / factorial(n+1)
        print(f"{x0:6.1f} {n:3d} {approx:14.10f} {exact:14.10f} "
              f"{err:12.3e} {bound:12.3e} {'OK' if err <= bound else '突破!':>8}")

# 界 vs 實際誤差
ns = np.arange(1, 16, 2)
x0 = 1.0
errs   = [abs(taylor_sin(x0, n) - math.sin(x0)) for n in ns]
bounds = [x0**(n+1) / factorial(n+1) for n in ns]
plt.semilogy(ns, errs, 'o-', label='actual error')
plt.semilogy(ns, bounds, 's--', label='Lagrange bound')
plt.xlabel('n'); plt.ylabel('error at x = 1'); plt.legend()
plt.title('The bound is always above the error — and close')
plt.show()

In [ ]:
# TODO 學生練習:把 sin 換成 cos(係數改 (-1)^k * x^(2k) / (2k)!)
# 對 x = 1,要取到幾階誤差才低於 1e-10?

## Lab 3｜自己寫一個 sin:區間縮減 + Horner

觀念 9 說函式庫用「縮區間 + 短多項式」。這格真的把它寫出來,和 <code>math.sin</code> 比。


In [ ]:
TWO_PI = 2 * math.pi

def my_sin(x0):
    """階段一:區間縮減;階段二:7 次多項式(Horner)"""
    # --- 縮到 [-pi, pi] ---
    x0 = x0 - TWO_PI * round(x0 / TWO_PI)
    # --- 再用對稱性縮到 [-pi/2, pi/2] ---
    if x0 > math.pi / 2:
        x0 = math.pi - x0
    elif x0 < -math.pi / 2:
        x0 = -math.pi - x0
    # --- Horner 形式的 7 次泰勒 ---
    x2 = x0 * x0
    return x0 * (1 + x2 * (-1/6 + x2 * (1/120 - x2 / 5040)))

print(f"{'x':>10} {'my_sin':>18} {'math.sin':>18} {'誤差':>12}")
for x0 in [0.1, 0.5, 1.0, 3.0, 10.0, 1000.0]:
    m, t = my_sin(x0), math.sin(x0)
    print(f"{x0:10.1f} {m:18.12f} {t:18.12f} {abs(m-t):12.3e}")

xs = np.linspace(-20, 20, 800)
err = [abs(my_sin(v) - math.sin(v)) for v in xs]
plt.semilogy(xs, np.maximum(err, 1e-18))
plt.xlabel('x'); plt.ylabel('|my_sin - math.sin|')
plt.title('Range reduction keeps the error flat everywhere')
plt.show()
print("\n最大誤差 =", max(err))

In [ ]:
# TODO 學生練習:把 my_sin 的多項式砍到 5 次(去掉 x2/5040 那項)
# 最大誤差變成多少?符合觀念 8 的誤差表嗎?

## Lab 4｜泰勒證明的預測:中央差分真的是 O(h²) 嗎

觀念 10 用泰勒算出前向差分是 $O(h)$、中央差分是 $O(h^2)$,最佳步長分別是 $\sqrt{\varepsilon}$ 與 $\varepsilon^{1/3}$。這格檢驗這些預測。


In [ ]:
f, df = math.sin, math.cos
x0 = 1.0
exact = df(x0)

hs = np.array([10.0**(-k) for k in np.arange(1, 16, 0.5)])
fwd = np.array([abs((f(x0+h) - f(x0))/h - exact) for h in hs])
ctr = np.array([abs((f(x0+h) - f(x0-h))/(2*h) - exact) for h in hs])

plt.loglog(hs, np.maximum(fwd, 1e-18), 'o-', label='forward  O(h)')
plt.loglog(hs, np.maximum(ctr, 1e-18), 's-', label='central  O(h^2)')
eps = np.finfo(float).eps
plt.axvline(np.sqrt(eps),   color='C0', ls='--', label='sqrt(eps)')
plt.axvline(eps**(1/3),     color='C1', ls=':',  label='eps^(1/3)')
plt.gca().invert_xaxis(); plt.xlabel('h'); plt.ylabel('|error|'); plt.legend(fontsize=8)
plt.title('Taylor predicts both the slope and the optimum')
plt.show()

print("前向差分最佳 h =", f"{hs[np.argmin(fwd)]:.2e}", "  預測 sqrt(eps) =", f"{np.sqrt(eps):.2e}")
print("中央差分最佳 h =", f"{hs[np.argmin(ctr)]:.2e}", "  預測 eps^(1/3) =", f"{eps**(1/3):.2e}")

# 檢查斜率:在截斷誤差主導的區段,log-log 斜率應為 1 與 2
big = hs > 1e-4
for name, e in [('forward', fwd), ('central', ctr)]:
    slope = np.polyfit(np.log10(hs[big]), np.log10(e[big]), 1)[0]
    print(f"{name} 在大 h 區段的 log-log 斜率 = {slope:.3f}")